# Faruq-v3 — leaf-rank headroom audit

Validation-only inference pada checkpoint D0. Mengukur apakah kelas benar masih berada pada top-2/top-3 ketika top-1 salah. Tidak melakukan training dan tidak mengakses test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import tarfile
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-leaf-rank-headroom-v1/leaf_rank_headroom.json'
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive: archive.extractall('/content', filter='data')
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('PROJECT   :', PROJECT_ROOT)
print('CHECKPOINT:', CHECKPOINT)
print('OUTPUT    :', OUTPUT)

In [ ]:
from coffee_detector.analysis.faruq_v3_leaf_rank_headroom import run_faruq_v3_leaf_rank_headroom
result = run_faruq_v3_leaf_rank_headroom(
    CHECKPOINT, DATA_ROOT, OUTPUT, device='0', image_size=640, candidate_count=500, iou_threshold=0.50
)
assert result['training_executed'] is False
assert result['validation_images_accessed'] is True
assert result['test_images_accessed'] is False
print('AUDIT SELESAI')

In [ ]:
import pandas as pd
from IPython.display import display
headline = {key: result['global'][key] for key in ('targets', 'matched', 'proposal_accessibility', 'matched_recall', 'conditional_top1_accuracy', 'conditional_top2_accuracy', 'conditional_top3_accuracy', 'conditional_top5_accuracy', 'top3_recovery_over_top1', 'mean_reciprocal_rank', 'median_true_class_rank')}
display(pd.DataFrame([headline]).style.format({key: '{:.2%}' for key in ('proposal_accessibility', 'matched_recall', 'conditional_top1_accuracy', 'conditional_top2_accuracy', 'conditional_top3_accuracy', 'conditional_top5_accuracy', 'top3_recovery_over_top1', 'mean_reciprocal_rank')}))
per_class = pd.DataFrame(result['per_class']).sort_values(['conditional_top1_accuracy', 'class_name'], na_position='last')
display(per_class.head(10).style.format({'conditional_top1_accuracy': '{:.2%}', 'conditional_top3_accuracy': '{:.2%}', 'top3_recovery_over_top1': '{:+.2%}'}))
display(pd.DataFrame(result['top_confusion_pairs']).head(15).style.format({'true_class_top3_fraction': '{:.2%}', 'median_true_minus_other_margin': '{:+.4f}'}))
print('DECISION:', result['decision']['decision'])
print('NEXT:', result['decision']['next_action'])
print('TRAINING AUTHORIZED:', result['decision']['training_authorized'])
print('SUMMARY:', result['summary'])
print('Kirim headline, bottom-10, confusion pairs, dan keputusan. Jangan training.')